# PASC Decoding Pipeline

Pipeline logic with the following steps per decoding step:
1. **PROBE**: Read attention from layer 18 (PAS signal) and 22 (localization).
2. **CANDIDATE SET**: Nucleus top-p=0.9 of next token distribution.
3. **EVIDENCE RESCORING**: Rescore distribution using evidence bank if token is worth checking.
4. **TRIGGER**: Gate mode `either`. Fires when model hesitates (gap < 0.2, k >= 2) OR ignores image (PAS z-score > 1.5).
5. **CORRECT**: Crop based on attention, prompt base model with crop, get visual evidence and new token.

In [ ]:
import json
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# File can be updated here
RESULT_FILE = "../results/treebench_pasc_pasc_20260811_174229.json"

# Configs
GATE_MODE = "either"
TOP_P = 0.9

In [ ]:
class PASCPipeline:
    def __init__(self, model, evidence_bank):
        self.model = model
        self.evidence_bank = evidence_bank
        self.cooldown = 0
        
    def decode_step(self, input_ids, image, image_baseline):
        """
        Mỗi decoding step:
        """
        # 1. PROBE (attn_probe.py) — đọc 1 hàng attention từ 2 layer (18 + 22)
        # PAS signal (layer 18): model đang bám text hay bám ảnh
        # localization row (layer 22): NHÌN vào đâu trong ảnh
        pas_signal, img_attn = self.probe_attention(input_ids, image, layers=[18, 22])
        pas_zscore = self.compute_zscore(pas_signal)
        
        # Get next token distribution
        logits = self.model(input_ids)
        probs = torch.softmax(logits, dim=-1)
        
        # 2. CANDIDATE SET — nucleus top-p=0.9 của phân phối token kế tiếp
        candidates = self.get_nucleus_set(probs, top_p=TOP_P)
        
        # 3. EVIDENCE RESCORING — nếu token "đáng kiểm tra" (không phải stopword/dấu câu):
        # trộn phân phối gốc với điểm nhất quán từ evidence bank
        top_token = candidates.top_token()
        if self.is_worth_checking(top_token):
            probs = self.evidence_bank.rescore(probs)
            candidates = self.get_nucleus_set(probs, top_p=TOP_P)
            top_token = candidates.top_token()
            
        # 4. TRIGGER (gate_mode="either") — bắn khi:
        # (gap < 0.2 VÀ k >= 2)          <- model đang phân vân
        # HOẶC (PAS z-score > 1.5)       <- model tự tin nhưng KHÔNG nhìn ảnh
        # + token groundable + qua cooldown
        gap, k = self.calculate_gap_and_k(candidates)
        is_hesitating = (gap < 0.2 and k >= 2)
        is_ignoring_image = (pas_zscore > 1.5)
        
        trigger_fired = (is_hesitating or is_ignoring_image) and \
                        self.is_groundable(top_token) and \
                        self.cooldown <= 0
                        
        if not trigger_fired:
            self.cooldown -= 1
            return top_token
            
        # 5. NẾU BẮN -> CORRECT (rel_attention.py + self_correct.py)
        # crop = (img_attn / baseline, có floor) quanh vùng attention cao
        crop_mask = np.maximum(img_attn / image_baseline, 0.1) # có floor
        crop = self.apply_crop(image, crop_mask)
        
        # hỏi lại base model với [ảnh gốc + crop]
        # model chọn 1 token (được phép ngoài nucleus, theo rank) + nêu 1 câu visual evidence
        new_token, visual_evidence = self.ask_base_model_with_crop(input_ids, image, crop)
        
        # force token nếu đổi + thêm evidence vào bank
        if new_token != top_token:
            self.evidence_bank.add(visual_evidence)
            top_token = new_token
            
        self.cooldown = 5 # reset cooldown
        return top_token
        
    # Dummy helper methods for illustration
    def probe_attention(self, ids, img, layers): return 0.0, np.ones((10,10))
    def compute_zscore(self, signal): return 0.0
    def get_nucleus_set(self, probs, top_p): return type('Mock', (object,), {'top_token': lambda s: 'token'})()
    def is_worth_checking(self, token): return True
    def calculate_gap_and_k(self, candidates): return 0.1, 3
    def is_groundable(self, token): return True
    def apply_crop(self, img, mask): return img
    def ask_base_model_with_crop(self, ids, img, crop): return 'new_token', 'evidence sentence'


In [ ]:
def visualize_correction(item):
    """
    Ít markdown: Show ảnh được crop và token sửa
    """
    print("="*50)
    print(f"Token thay đổi: {item.get('original_token', 'N/A')} -> {item.get('corrected_token', 'N/A')}")
    print(f"Lý do (Visual Evidence): {item.get('evidence', 'N/A')}")
    
    crop_path = item.get('crop_path')
    if crop_path and os.path.exists(crop_path):
        try:
            img = Image.open(crop_path)
            plt.figure(figsize=(4,4))
            plt.imshow(img)
            plt.title("Cropped Area")
            plt.axis('off')
            plt.show()
        except Exception as e:
            print(f"Không thể load ảnh {crop_path}: {e}")
    else:
        print("[Không có ảnh crop để hiển thị]")

def process_results(filepath):
    if not os.path.exists(filepath):
        print(f"Không tìm thấy file: {filepath}")
        print("Tạo dummy data để demo...")
        # Dummy data cho demo
        dummy_data = [{
            "original_token": "car",
            "corrected_token": "truck",
            "evidence": "The vehicle has a large cargo bed at the back.",
            "crop_path": "",
            "corrected": True
        }]
        results = dummy_data
    else:
        with open(filepath, 'r') as f:
            results = json.load(f)
            
    for item in results:
        if item.get("corrected", False):
            visualize_correction(item)

# Chạy pipeline hiển thị với file kết quả
process_results(RESULT_FILE)


AttributeError: 'str' object has no attribute 'get'